# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #4** — **"The Freshness Multiplier"** claims 365+ day content refreshed within 30 days shows a 3.2x health boost and 57x more impressions, and reports a 283:1 growth-to-decline ratio for the 361+ freshness bucket.

*Methodology question:* Where does the label come from? `trend_direction` (growing vs. declining) is a defined rule — 30d-vs-prev-30d impression change — not an independently observed outcome, which the paper is transparent about elsewhere. My real question is about the 283:1 ratio specifically: the paper itself flags that this cell has only 1 declining page in the bucket, which means the ratio is almost entirely a function of how few pages fall in that denominator, not a stable multiplier. Does the validation design carry the claim? The paper handles this well — it explicitly demotes the 283:1 number and labels it "unstable," and it separates the more defensible 3.13:1 ratio for the 181-360 bucket. That's the right instinct; the constructive follow-up question is why the headline 3.2x/57x refresh-boost numbers (from the age-freshness matrix) don't get the same small-sample caveat treatment, since the "365+ × 361+" quadrant is explicitly called out elsewhere in the paper as having "strong survivor bias" from the active-content filter (impressions_90d > 0 and sessions_90d > 0). If a page fully died, it wouldn't be in this sample at all — so the refresh-boost comparison may be comparing "recovered survivors" against "not-yet-recovered survivors," not against pages that failed to recover and dropped out of visibility entirely.

**Finding #10 — "AI Model Performance"** compares OpenAI vs. Gemini cohorts and lands on a nuanced, nowinner-declared conclusion — which is itself a well-handled finding. My question is about the Random Forest feature importance appendix that sits nearby: Average Position is reported as the top predictor of `health_score` at 43% importance. Where does the label come from? The paper states directly that `health_score` is constructed from position, impressions, CTR, and scroll depth (30/30/20/20 pts) — so `avg_position` is a direct input to the label it's "predicting." That's the exact label-derived-feature pattern the leakage skill calls the confession symptom: one feature towering over the rest, paired with a target the feature helped compute. Does the validation design carry the claim? The paper is honest about this too — it says explicitly "importance is descriptive rather than causal" and that "high importance is therefore expected." The constructive question worth raising: since `avg_position` + `impressions` together are 75% of the importance and both are label components, does reporting this appendix as "What Predicts Health?" (a causal-sounding title) undersell how much of that 75% is circular by construction? A reader skimming just the header and bar chart could easily walk away thinking position "drives" health in a discoverable, actionable way, rather than "position is literally a third of how health is scored."

Both are offered in the spirit the paper itself sets — it already does a lot of this self-auditing (flagging survivor bias, labeling importance as descriptive not causal), so these are "push the same rigor one notch further," not gotchas.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Let's rebuild this cleanly and run the actual comparison, since your Week-5 notebook's model+split state won't have survived into this new one (same issue as before — recreate everything fresh so it runs top-to-bottom).

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

feature_cols = [
    "search_volume", "competition", "cpc", "word_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "engaged_sessions_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "content_age_days", "days_since_last_update"
]

X = df[feature_cols].copy()
for col in ["search_volume", "competition", "cpc", "word_count"]:
    X[f"has_{col}"] = df[col].notna().astype(int)
    X[col] = X[col].fillna(0)
X["scroll_rate"] = X["scroll_rate"].fillna(X["scroll_rate"].median())
feature_cols = feature_cols + ["has_search_volume", "has_competition", "has_cpc", "has_word_count"]

assert X.isna().sum().sum() == 0, "NaN still present — stop and fix before modeling"

# --- BEFORE: random split (what Week 5 likely used) ---
X_train_r, X_holdout_r = train_test_split(X, test_size=0.2, random_state=42)
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_holdout_r_scaled = scaler_r.transform(X_holdout_r)

km_r = KMeans(n_clusters=4, random_state=42, n_init=10)
km_r.fit(X_train_r_scaled)
sil_random = silhouette_score(X_holdout_r_scaled, km_r.predict(X_holdout_r_scaled))

# --- AFTER: grouped split by client_id (honest) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
fit_idx, holdout_idx = next(gss.split(X, groups=df["client_id"]))
X_fit_g, X_holdout_g = X.iloc[fit_idx], X.iloc[holdout_idx]

scaler_g = StandardScaler()
X_fit_g_scaled = scaler_g.fit_transform(X_fit_g)
X_holdout_g_scaled = scaler_g.transform(X_holdout_g)

km_g = KMeans(n_clusters=4, random_state=42, n_init=10)
km_g.fit(X_fit_g_scaled)
sil_grouped = silhouette_score(X_holdout_g_scaled, km_g.predict(X_holdout_g_scaled))

print(f"| Split type | Holdout silhouette |")
print(f"|---|---|")
print(f"| Random split (BEFORE) | {sil_random:.3f} |")
print(f"| Grouped-by-client split (AFTER) | {sil_grouped:.3f} |")

| Split type | Holdout silhouette |
|---|---|
| Random split (BEFORE) | 0.294 |
| Grouped-by-client split (AFTER) | 0.278 |


Under a random 80/20 split, the K-Means model (k=4) achieved a holdout silhouette of 0.294. Under an honest grouped-by-client split — where no client appears in both the fit and holdout sets — the holdout silhouette was 0.278, a drop of 0.016 (~5% relative). This gap is the honest cost of the random split's optimism: some of the apparent cluster quality in the random-split number came from the model having already seen that client's pages during fitting, not from patterns that generalize to entirely new clients. The gap is real but modest, meaning the clusters capture some genuine cross-client structure rather than being purely a memorized client signature — but it also means client identity is not fully separable from the behavioral features being clustered on. The grouped number (0.278) is the one that should be reported and trusted going forward, since it's the only one that answers "does this work on a client the model hasn't seen."

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# --- Leakage checklist, run against the final feature set above ---

# 1. Label-derived features: is trend_direction/trend_pct anywhere in feature_cols?
label_derived = [c for c in feature_cols if c in ["trend_direction", "trend_pct", "is_declining_label"]]
print("Label-derived fields in feature set (should be empty):", label_derived)

# 2. Product-decision flags: provider_used / model_used
product_flags = [c for c in feature_cols if c in ["provider_used", "model_used"]]
print("Product-decision flags in feature set (should be empty):", product_flags)

# 3. Context/ID columns used as features
id_cols = [c for c in feature_cols if c in ["content_id", "client_id"]]
print("ID columns in feature set (should be empty):", id_cols)

# 4. Future/overlapping windows: confirm every feature is backward-looking only
# (manual check — list what's actually in feature_cols)
print("\nFinal feature set:", feature_cols)

# 5. Grouped split confirmed (from section 2)
overlap = set(df.iloc[fit_idx]["client_id"]) & set(df.iloc[holdout_idx]["client_id"])
print("\nClient overlap between fit/holdout (should be empty):", overlap)

# 6. Deliberately inject a leaky feature and confirm the harness reacts
X_leaky = X_fit_g.copy()
leaked = df.loc[X_fit_g.index, "trend_pct"]
print("NaN count in trend_pct for this slice:", leaked.isna().sum())
X_leaky["leaked_trend_pct"] = leaked.fillna(leaked.median())  # fill just for this deliberate test

X_leaky_scaled = StandardScaler().fit_transform(X_leaky)
km_leak_test = KMeans(n_clusters=4, random_state=42, n_init=10)
sil_leak_test = silhouette_score(X_leaky_scaled, km_leak_test.fit_predict(X_leaky_scaled))
print(f"\nSilhouette WITH deliberately leaked trend_pct: {sil_leak_test:.3f}")
print(f"Silhouette WITHOUT (honest, grouped): {sil_grouped:.3f}")
print("If the leaked version scores meaningfully higher, the harness is correctly sensitive to leakage.")

Label-derived fields in feature set (should be empty): []
Product-decision flags in feature set (should be empty): []
ID columns in feature set (should be empty): []

Final feature set: ['search_volume', 'competition', 'cpc', 'word_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'content_age_days', 'days_since_last_update', 'has_search_volume', 'has_competition', 'has_cpc', 'has_word_count']

Client overlap between fit/holdout (should be empty): set()
NaN count in trend_pct for this slice: 2704

Silhouette WITH deliberately leaked trend_pct: 0.293
Silhouette WITHOUT (honest, grouped): 0.278
If the leaked version scores meaningfully higher, the harness is correctly sensitive to leakage.


The final feature set (19 fields, including four `has_-flags` for structurally missing data) was audited against the full leakage checklist: no label-derived fields (`trend_direction, trend_pct, is_declining_label`), no product-decision flags (`provider_used, model_used`), and no ID columns (`content_id, client_id`) are present as features. The grouped fit/holdout split has zero client overlap, confirmed directly.

To verify the audit harness itself was sensitive to leakage rather than just rubber-stamping a clean-looking feature list, `trend_pct` — a field known to be derived the same way as the excluded `trend_direction` label — was deliberately injected as a test feature. Silhouette rose from 0.278 (honest, grouped) to 0.293 (with the leaked feature), a measurable increase, confirming the harness correctly detects when a leaky signal is added. This is comparable in size to the random-vs-grouped split gap found in section 2 (0.016), suggesting client memorization and label-adjacent leakage are both modest but real sources of inflated apparent cluster quality in this dataset — neither dominates, but both matter enough to guard against.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In this snapshot, K-Means clustering (k=4, holdout silhouette 0.278 under a grouped-by-client split) found some structure beyond simple thresholds — most clearly a small high-performing cluster (382 pages, median 503 sessions, 2.63% engagement) and a distinct low-demand cluster (2,469 pages, median 3 sessions). However, the remaining two clusters were driven predominantly by `has_word_count`, a near-perfect proxy for `content_type`, rather than by independent behavioral signal. This is a directional, decision-support finding, not proof that a fixed number of true behavioral archetypes exists in the data — part of what the clustering recovered was closer to known content categories than to a new pattern, and that should be stated plainly rather than folded into a broader 'archetypes are real' claim."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.